In [1]:
%reload_ext autoreload
%autoreload 2

In [1]:
# ----------------------- I M P O R T S --------------------------- #

from dataforge.scripts.build_nmers import build_nmers

# ----------------------- C O N F I G S --------------------------- #

INPUT_FILENAME = "/storage_common/angiod/MB-Fit-Data-Forge/POPC/traj/prod.T303.T353.T403.T453.npz"
# INPUT_FILENAME = "/storage_common/angiod/MB-Fit-Data-Forge/POPC/traj/prod-stride-17.npz"
DATASET_ROOT   = "/storage_common/angiod/MB-Fit-Data-Forge/POPC/POPC.MULTITEMP/"

# --- Option 1 --- #
# k: dict(n=N, method=US)  = Uniform Sampling (Default)
# k: dict(n=N, method=FPS) = Furthest Point Sampling

# NMER_SAMPLING_CONF = {
#     1: 100, # Default method is US
#     2: 100, # Default method is US
#     3: dict(n=1, method='FPS')
# }

# --- Option 2 --- #

# Does not perform sampling, takes all n-mers in the dataset.
# This method is faster cause it does not compute descriptors,
# but can be memory consuming
NMER_SAMPLING_CONF = [1, 2, 3]

# ----------------------------------------------------------------- #

In [ ]:
build_nmers(
    input_filename = INPUT_FILENAME,
    dataset_root   = DATASET_ROOT,
    nmer_sampling_conf = NMER_SAMPLING_CONF,
    # keep_only_monomer_names=['C-CCHO', 'C__OO-CC'], # ['C-CHHO', 'C-CCHO', 'C__OO-CC', 'C-CCHH'],
    max_processes=32,
)

In [57]:
from dataforge.scripts.build_nmers import prepare_qchem_input
from dataforge.src import DataDict
from os.path import join

DATA_ROOT = "/storage_common/angiod/MB-Fit-Data-Forge/POPC/POPC.DELETE/data"
NMERS_CAPPED_ROOT = join(DATA_ROOT, "xyz_capped/")
QCHEM_IN_ROOT     = join(DATA_ROOT, "qchem_input/")
QCHEM_MIN_IN_ROOT = join(DATA_ROOT, "qchem_min_input/")

prepare_qchem_input(
    NMERS_CAPPED_ROOT,
    QCHEM_IN_ROOT,
    QCHEM_MIN_IN_ROOT,
    DataDict.CHARGES_DICT,
    max_processes=0,
)

In [ ]:
import os
import numpy as np
from dataforge.scripts.fps import furthest_point_sampling
from dataforge.src.generic import read_h5_file

h5_filename = '/storage_common/angiod/MB-Fit-Data-Forge/POPC/POPC.MULTITEMP/data/xyz_capped/trimers/C-CHHO.C-CHHO.POOOO-CC|0_1H.0_1H.1_00/C-CHHO.C-CHHO.POOOO-CC|0_1H.0_1H.1_00_36_37_0.h5'
N_SAMPLES = 5000

coords,atom_types,fullnames,info_dict,extra_data = read_h5_file(h5_filename)
names = extra_data['symmetry_names_sorted']


sampled_indices = furthest_point_sampling(N_SAMPLES, coords, names, chunk_max_dim=10000, max_processes=1)

def get_list_filename(h5_filename):
    base, _ = os.path.splitext(h5_filename)
    return base + '.list'


output_filepath = get_list_filename(h5_filename)
np.savetxt(output_filepath, sampled_indices, fmt='%d')